In [0]:
print("Shadow Revenue Detection System")
print("Spark Version:", spark.version)

Shadow Revenue Detection System
Spark Version: 4.1.0


In [0]:
data = [
    (1, "Order_1", 100.50),
    (2, "Order_2", 250.75),
    (3, "Order_3", 500.00)
]

test_df = spark.createDataFrame(
    data,
    ["order_id", "order_name", "amount"]
)

display(test_df)

order_id,order_name,amount
1,Order_1,100.5
2,Order_2,250.75
3,Order_3,500.0


In [0]:
test_df.createOrReplaceTempView("test_orders")

In [0]:
%sql
SELECT *
FROM test_orders;

order_id,order_name,amount
1,Order_1,100.5
2,Order_2,250.75
3,Order_3,500.0


In [0]:
print("Shadow Revenue Detection System")
print("Spark Version:", spark.version)

Shadow Revenue Detection System
Spark Version: 4.1.0


In [0]:
# ============================================================
# SHADOW REVENUE DETECTION SYSTEM
# PROJECT CONFIGURATION
# ============================================================

# Dataset sizes
NUM_PRODUCTS = 100
NUM_ORDERS = 10000
NUM_CUSTOMERS = 1000

# Anomaly injection rates
DUPLICATE_RATE = 0.02
MISSING_PAYMENT_RATE = 0.10
ORPHAN_PAYMENT_RATE = 0.03
PRICE_MISMATCH_RATE = 0.10

print("Project configuration loaded successfully.")
print("----------------------------------------")
print(f"Products           : {NUM_PRODUCTS}")
print(f"Orders             : {NUM_ORDERS}")
print(f"Customers          : {NUM_CUSTOMERS}")
print(f"Duplicate Rate     : {DUPLICATE_RATE * 100}%")
print(f"Missing Payment    : {MISSING_PAYMENT_RATE * 100}%")
print(f"Orphan Payment     : {ORPHAN_PAYMENT_RATE * 100}%")
print(f"Price Mismatch     : {PRICE_MISMATCH_RATE * 100}%")

Project configuration loaded successfully.
----------------------------------------
Products           : 100
Orders             : 10000
Customers          : 1000
Duplicate Rate     : 2.0%
Missing Payment    : 10.0%
Orphan Payment     : 3.0%
Price Mismatch     : 10.0%


In [0]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

from pyspark.sql import Row
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

import random

print("Libraries imported successfully.")

Libraries imported successfully.


In [0]:
# ============================================================
# CUSTOMER DATA GENERATION
# ============================================================

customers = []

cities = [
    "Delhi",
    "Mumbai",
    "Bangalore",
    "Pune",
    "Hyderabad",
    "Dehradun"
]

for customer_id in range(1, NUM_CUSTOMERS + 1):

    customers.append(
        Row(
            customer_id=customer_id,
            customer_name=f"Customer_{customer_id}",
            city=random.choice(cities)
        )
    )

customer_df = spark.createDataFrame(customers)

print("Customer dataset created.")
print("Total customers:", customer_df.count())

display(customer_df.limit(10))

Customer dataset created.
Total customers: 1000


customer_id,customer_name,city
1,Customer_1,Dehradun
2,Customer_2,Pune
3,Customer_3,Dehradun
4,Customer_4,Pune
5,Customer_5,Bangalore
6,Customer_6,Delhi
7,Customer_7,Mumbai
8,Customer_8,Dehradun
9,Customer_9,Delhi
10,Customer_10,Dehradun


In [0]:
# ============================================================
# PRODUCT DATA GENERATION - SCD TYPE 2
# ============================================================

products = []

for product_id in range(1, NUM_PRODUCTS + 1):

    # Original product price
    base_price = round(
        random.uniform(10, 500),
        2
    )

    # --------------------------------------------------------
    # OLD PRODUCT RECORD
    # --------------------------------------------------------

    products.append(
        Row(
            product_id=product_id,
            price=base_price,
            effective_date="2023-01-01",
            end_date="2023-12-31",
            is_current=0
        )
    )

    # --------------------------------------------------------
    # CURRENT PRODUCT RECORD
    # --------------------------------------------------------

    current_price = round(
        base_price * random.uniform(0.9, 1.2),
        2
    )

    products.append(
        Row(
            product_id=product_id,
            price=current_price,
            effective_date="2024-01-01",
            end_date=None,
            is_current=1
        )
    )


# Create Spark DataFrame
product_df = spark.createDataFrame(products)

print("Product dataset created.")
print("Total product records:", product_df.count())

display(product_df.orderBy("product_id").limit(20))

---------------------------------------------------------------------------
PySparkTypeError                          Traceback (most recent call last)
File <command-4619407234214452>, line 10
      5 products = []
      7 for product_id in range(1, NUM_PRODUCTS + 1):
      8 
      9     # Original product price
---> 10     base_price = round(
     11         random.uniform(10, 500),
     12         2
     13     )
     15     # --------------------------------------------------------
     16     # OLD PRODUCT RECORD
     17     # --------------------------------------------------------
     19     products.append(
     20         Row(
     21             product_id=product_id,
   (...)
     26         )
     27     )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/utils.py:308, in try_remote_functions.<locals>.wrapped(*args, **kwargs)
    305 if is_remote() and "PYSPARK_NO_NAMESPACE_SHARE" not in os.environ:
    306     from pyspark.sql.connect import functions
--> 

In [0]:
# ============================================================
# FIX SPARK COLUMN FUNCTION
# ============================================================

from pyspark.sql.functions import col

print("Spark column function restored.")

Spark column function restored.


In [0]:
# ============================================================
# PRODUCT DATA GENERATION - SCD TYPE 2
# ============================================================

products = []

for product_id in range(1, NUM_PRODUCTS + 1):

    # Generate original product price
    base_price = round(
        random.uniform(10, 500),
        2
    )

    # Historical / expired record
    products.append(
        Row(
            product_id=product_id,
            price=base_price,
            effective_date="2023-01-01",
            end_date="2023-12-31",
            is_current=0
        )
    )

    # Current / active record
    current_price = round(
        base_price * random.uniform(0.9, 1.2),
        2
    )

    products.append(
        Row(
            product_id=product_id,
            price=current_price,
            effective_date="2024-01-01",
            end_date=None,
            is_current=1
        )
    )

# Create Spark DataFrame
product_df = spark.createDataFrame(products)

print("Product dataset created successfully.")
print("Total product records:", product_df.count())

---------------------------------------------------------------------------
PySparkTypeError                          Traceback (most recent call last)
File <command-4619407234214454>, line 10
      5 products = []
      7 for product_id in range(1, NUM_PRODUCTS + 1):
      8 
      9     # Generate original product price
---> 10     base_price = round(
     11         random.uniform(10, 500),
     12         2
     13     )
     15     # Historical / expired record
     16     products.append(
     17         Row(
     18             product_id=product_id,
   (...)
     23         )
     24     )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/utils.py:308, in try_remote_functions.<locals>.wrapped(*args, **kwargs)
    305 if is_remote() and "PYSPARK_NO_NAMESPACE_SHARE" not in os.environ:
    306     from pyspark.sql.connect import functions
--> 308     return getattr(functions, f.__name__)(*args, **kwargs)
    309 else:
    310     return f(*args, **kwargs)

File /da

In [0]:
# Fix Spark functions safely
from pyspark.sql import functions as F

print("Spark functions loaded successfully.")

Spark functions loaded successfully.


In [0]:
# ============================================================
# PRODUCT DATA GENERATION - SCD TYPE 2
# ============================================================

products = []

for product_id in range(1, NUM_PRODUCTS + 1):

    # Historical price
    base_price = round(
        random.uniform(10, 500),
        2
    )

    # Old / historical record
    products.append(
        Row(
            product_id=product_id,
            price=base_price,
            effective_date="2023-01-01",
            end_date="2023-12-31",
            is_current=0
        )
    )

    # Current price
    current_price = round(
        base_price * random.uniform(0.9, 1.2),
        2
    )

    # Current / active record
    products.append(
        Row(
            product_id=product_id,
            price=current_price,
            effective_date="2024-01-01",
            end_date=None,
            is_current=1
        )
    )

# Create Spark DataFrame
product_df = spark.createDataFrame(products)

print("Product dataset created successfully.")
print("Total product records:", product_df.count())

---------------------------------------------------------------------------
PySparkTypeError                          Traceback (most recent call last)
File <command-4619407234214456>, line 10
      5 products = []
      7 for product_id in range(1, NUM_PRODUCTS + 1):
      8 
      9     # Historical price
---> 10     base_price = round(
     11         random.uniform(10, 500),
     12         2
     13     )
     15     # Old / historical record
     16     products.append(
     17         Row(
     18             product_id=product_id,
   (...)
     23         )
     24     )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/utils.py:308, in try_remote_functions.<locals>.wrapped(*args, **kwargs)
    305 if is_remote() and "PYSPARK_NO_NAMESPACE_SHARE" not in os.environ:
    306     from pyspark.sql.connect import functions
--> 308     return getattr(functions, f.__name__)(*args, **kwargs)
    309 else:
    310     return f(*args, **kwargs)

File /databricks/python/lib

In [0]:
from pyspark.sql.functions import *

In [0]:
# ============================================================
# PRODUCT DATA GENERATION - SCD TYPE 2
# FIXED VERSION
# ============================================================

import builtins

products = []

for product_id in range(1, NUM_PRODUCTS + 1):

    # Generate historical/base price
    base_price = builtins.round(
        random.uniform(10, 500),
        2
    )

    # --------------------------------------------------------
    # HISTORICAL RECORD
    # --------------------------------------------------------

    products.append(
        Row(
            product_id=product_id,
            price=base_price,
            effective_date="2023-01-01",
            end_date="2023-12-31",
            is_current=0
        )
    )

    # --------------------------------------------------------
    # CURRENT RECORD
    # --------------------------------------------------------

    current_price = builtins.round(
        base_price * random.uniform(0.9, 1.2),
        2
    )

    products.append(
        Row(
            product_id=product_id,
            price=current_price,
            effective_date="2024-01-01",
            end_date=None,
            is_current=1
        )
    )

# Create Spark DataFrame
product_df = spark.createDataFrame(products)

print("Product dataset created successfully.")
print("Total product records:", product_df.count())

Product dataset created successfully.
Total product records: 200


In [0]:
# ============================================================
# VERIFY PRODUCT SCD TYPE 2
# ============================================================

historical_count = product_df.filter(
    F.col("is_current") == 0
).count()

current_count = product_df.filter(
    F.col("is_current") == 1
).count()

print("Historical records:", historical_count)
print("Current records:", current_count)
print("Total records:", product_df.count())

Historical records: 100
Current records: 100
Total records: 200


In [0]:
display(product_df.limit(20))

product_id,price,effective_date,end_date,is_current
1,267.3,2023-01-01,2023-12-31,0
1,306.5,2024-01-01,null,1
2,431.35,2023-01-01,2023-12-31,0
2,392.46,2024-01-01,null,1
3,476.1,2023-01-01,2023-12-31,0
3,555.13,2024-01-01,null,1
4,131.4,2023-01-01,2023-12-31,0
4,126.22,2024-01-01,null,1
5,468.4,2023-01-01,2023-12-31,0
5,514.02,2024-01-01,null,1


In [0]:
# ============================================================
# ORDER DATA GENERATION
# ============================================================

orders = []

for order_id in range(1, NUM_ORDERS + 1):

    # Random product and customer
    product_id = random.randint(1, NUM_PRODUCTS)
    customer_id = random.randint(1, NUM_CUSTOMERS)

    # Random quantity
    quantity = random.randint(1, 5)

    # Get the CURRENT catalog price for this product
    product_rows = [
        p for p in products
        if p.product_id == product_id
        and p.is_current == 1
    ]

    catalog_price = product_rows[0].price

    # --------------------------------------------------------
    # PRICE MISMATCH ANOMALY
    # Only ~10% of orders get an incorrect price
    # --------------------------------------------------------

    if random.random() < PRICE_MISMATCH_RATE:

        order_price = builtins.round(
            catalog_price * random.uniform(0.70, 1.30),
            4
        )

    else:

        # Correct catalog price
        order_price = builtins.round(
            catalog_price,
            4
        )

    # Create order
    orders.append(
        Row(
            order_id=order_id,
            product_id=product_id,
            customer_id=customer_id,
            quantity=quantity,
            price=order_price,
            order_date=f"2024-01-{random.randint(1, 28):02d}"
        )
    )

# Create Spark DataFrame
orders_df = spark.createDataFrame(orders)

print("Order dataset created successfully.")
print("Unique orders:", orders_df.count())

display(orders_df.limit(10))

Order dataset created successfully.
Unique orders: 10000


order_id,product_id,customer_id,quantity,price,order_date
1,98,617,3,196.8,2024-01-12
2,77,293,4,269.98,2024-01-04
3,31,388,1,426.07,2024-01-01
4,99,48,3,368.86,2024-01-04
5,91,316,5,330.13,2024-01-28
6,92,446,4,50.3,2024-01-13
7,20,1000,4,57.88,2024-01-09
8,46,578,3,162.64,2024-01-27
9,46,521,3,162.64,2024-01-26
10,37,723,1,209.49,2024-01-13


In [0]:
# ============================================================
# INJECT DUPLICATE ORDERS
# ============================================================

duplicate_count = int(
    NUM_ORDERS * DUPLICATE_RATE
)

# Randomly select existing orders
duplicate_orders = random.sample(
    orders,
    duplicate_count
)

# Add duplicates
orders_with_duplicates = (
    orders + duplicate_orders
)

# Create final DataFrame
orders_df = spark.createDataFrame(
    orders_with_duplicates
)

print("Duplicate records injected:", duplicate_count)
print("Total order records:", orders_df.count())

Duplicate records injected: 200
Total order records: 10200


In [0]:
# ============================================================
# VERIFY DUPLICATE ORDERS
# ============================================================

duplicate_check = (
    orders_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Order IDs having duplicates:",
    duplicate_check.count()
)

display(duplicate_check.limit(10))

Order IDs having duplicates: 200


order_id,count
398,2
975,2
730,2
78,2
669,2
184,2
419,2
1119,2
1200,2
513,2


In [0]:
# ============================================================
# VERIFY PRICE MISMATCH RATE - FIXED
# ============================================================

current_products = (
    product_df
    .filter(F.col("is_current") == 1)
    .select(
        "product_id",
        F.col("price").alias("catalog_price")
    )
)

order_price_check = (
    orders_df
    .dropDuplicates(["order_id"])
    .join(
        current_products,
        "product_id",
        "left"
    )
    .withColumn(
        "is_price_mismatch",
        F.when(
            F.col("price") != F.col("catalog_price"),
            1
        ).otherwise(0)
    )
)

mismatch_count = (
    order_price_check
    .filter(F.col("is_price_mismatch") == 1)
    .count()
)

total_unique_orders = order_price_check.count()

mismatch_percentage = (
    mismatch_count / total_unique_orders
) * 100

print("Unique orders:", total_unique_orders)
print("Price mismatches:", mismatch_count)
print(
    "Mismatch percentage:",
    builtins.round(mismatch_percentage, 2),
    "%"
)

Unique orders: 10000
Price mismatches: 1015
Mismatch percentage: 10.15 %


In [0]:
# ============================================================
# PAYMENT DATA GENERATION
# ============================================================

# Use the original 10,000 unique orders
unique_orders = orders

# Number of orders that should have missing payments
missing_payment_count = int(
    NUM_ORDERS * MISSING_PAYMENT_RATE
)

# Randomly select orders whose payments will be missing
missing_order_ids = set(
    random.sample(
        [order.order_id for order in unique_orders],
        missing_payment_count
    )
)

payments = []

payment_id = 1

# ------------------------------------------------------------
# CREATE LEGITIMATE PAYMENTS
# ------------------------------------------------------------

for order in unique_orders:

    # Skip selected orders to create missing payments
    if order.order_id in missing_order_ids:
        continue

    payment_amount = builtins.round(
        order.price * order.quantity,
        2
    )

    payments.append(
        Row(
            payment_id=payment_id,
            order_id=order.order_id,
            payment_amount=payment_amount,
            payment_status="SUCCESS"
        )
    )

    payment_id += 1


# ------------------------------------------------------------
# CREATE ORPHAN PAYMENTS
# ------------------------------------------------------------

ORPHAN_PAYMENT_COUNT = 300

for i in range(ORPHAN_PAYMENT_COUNT):

    # IDs that don't exist in the Orders table
    orphan_order_id = NUM_ORDERS + 1 + i

    payments.append(
        Row(
            payment_id=payment_id,
            order_id=orphan_order_id,
            payment_amount=builtins.round(
                random.uniform(50, 5000),
                2
            ),
            payment_status="SUCCESS"
        )
    )

    payment_id += 1


# Create Spark DataFrame
payments_df = spark.createDataFrame(payments)

print("Payment dataset created successfully.")
print("Legitimate payments:", NUM_ORDERS - missing_payment_count)
print("Missing payments:", missing_payment_count)
print("Orphan payments:", ORPHAN_PAYMENT_COUNT)
print("Total payment records:", payments_df.count())

Payment dataset created successfully.
Legitimate payments: 9000
Missing payments: 1000
Orphan payments: 300
Total payment records: 9300


In [0]:
display(payments_df.limit(20))

payment_id,order_id,payment_amount,payment_status
1,1,590.4,SUCCESS
2,2,1079.92,SUCCESS
3,3,426.07,SUCCESS
4,4,1106.58,SUCCESS
5,5,1650.65,SUCCESS
6,6,201.2,SUCCESS
7,7,231.52,SUCCESS
8,8,487.92,SUCCESS
9,9,487.92,SUCCESS
10,10,209.49,SUCCESS


In [0]:
# ============================================================
# VERIFY MISSING PAYMENTS
# ============================================================

order_ids_df = (
    spark.createDataFrame(
        [(order.order_id,) for order in unique_orders],
        ["order_id"]
    )
)

missing_payments_df = (
    order_ids_df
    .join(
        payments_df.select("order_id"),
        "order_id",
        "left_anti"
    )
)

missing_count = missing_payments_df.count()

print("Orders without payments:", missing_count)
print(
    "Missing payment percentage:",
    builtins.round(
        (missing_count / NUM_ORDERS) * 100,
        2
    ),
    "%"
)

Orders without payments: 1000
Missing payment percentage: 10.0 %


In [0]:
# ============================================================
# VERIFY ORPHAN PAYMENTS
# ============================================================

orphan_payments_df = (
    payments_df
    .join(
        order_ids_df,
        "order_id",
        "left_anti"
    )
)

orphan_count = orphan_payments_df.count()

print("Orphan payments:", orphan_count)

display(orphan_payments_df.limit(10))

Orphan payments: 300


order_id,payment_id,payment_amount,payment_status
10001,9001,3546.84,SUCCESS
10002,9002,251.62,SUCCESS
10003,9003,2332.58,SUCCESS
10004,9004,870.92,SUCCESS
10005,9005,4554.18,SUCCESS
10006,9006,591.56,SUCCESS
10007,9007,2826.5,SUCCESS
10008,9008,1213.1,SUCCESS
10009,9009,4453.45,SUCCESS
10010,9010,2003.62,SUCCESS


In [0]:
# ============================================================
# CHECK DATABRICKS CATALOG
# ============================================================

print("Current catalog:", spark.sql("SELECT current_catalog()").first()[0])
print("Current schema:", spark.sql("SELECT current_schema()").first()[0])

Current catalog: workspace
Current schema: default


In [0]:
# ============================================================
# BRONZE LAYER
# Save raw datasets as Delta tables
# ============================================================

# Bronze Customers
customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_customers")

# Bronze Products
product_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_products")

# Bronze Orders
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_orders")

# Bronze Payments
payments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_payments")

print("Bronze layer created successfully.")

Bronze layer created successfully.


In [0]:
# ============================================================
# VERIFY BRONZE LAYER
# ============================================================

bronze_tables = [
    "bronze_customers",
    "bronze_products",
    "bronze_orders",
    "bronze_payments"
]

print("BRONZE LAYER VALIDATION")
print("=" * 40)

for table in bronze_tables:
    count = spark.table(table).count()
    print(f"{table:<20} : {count} records")

BRONZE LAYER VALIDATION
bronze_customers     : 1000 records
bronze_products      : 200 records
bronze_orders        : 10200 records
bronze_payments      : 9300 records


In [0]:
# ============================================================
# SILVER LAYER - CLEAN ORDERS
# ============================================================

# Read orders from Bronze
bronze_orders_df = spark.table("bronze_orders")

print("Bronze order records:", bronze_orders_df.count())

# Remove duplicate orders using order_id
silver_orders_df = (
    bronze_orders_df
    .dropDuplicates(["order_id"])
)

print("Silver order records:", silver_orders_df.count())

print(
    "Duplicate records removed:",
    bronze_orders_df.count() - silver_orders_df.count()
)

Bronze order records: 10200
Silver order records: 10000
Duplicate records removed: 200


In [0]:
# ============================================================
# SILVER ORDERS - DATA QUALITY CHECKS
# ============================================================

null_order_ids = (
    silver_orders_df
    .filter(F.col("order_id").isNull())
    .count()
)

null_product_ids = (
    silver_orders_df
    .filter(F.col("product_id").isNull())
    .count()
)

null_customer_ids = (
    silver_orders_df
    .filter(F.col("customer_id").isNull())
    .count()
)

invalid_quantity = (
    silver_orders_df
    .filter(F.col("quantity") <= 0)
    .count()
)

invalid_price = (
    silver_orders_df
    .filter(F.col("price") <= 0)
    .count()
)

print("SILVER ORDERS - DATA QUALITY")
print("=" * 40)
print("Null order IDs      :", null_order_ids)
print("Null product IDs    :", null_product_ids)
print("Null customer IDs   :", null_customer_ids)
print("Invalid quantities  :", invalid_quantity)
print("Invalid prices      :", invalid_price)

SILVER ORDERS - DATA QUALITY
Null order IDs      : 0
Null product IDs    : 0
Null customer IDs   : 0
Invalid quantities  : 0
Invalid prices      : 0


In [0]:
# ============================================================
# SAVE SILVER ORDERS
# ============================================================

silver_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders")

print("silver_orders table created successfully.")

silver_orders table created successfully.


In [0]:
# Verify Silver Orders

silver_orders_check = spark.table("silver_orders")

print(
    "Silver Orders:",
    silver_orders_check.count(),
    "records"
)

display(
    silver_orders_check.limit(10)
)

Silver Orders: 10000 records


order_id,product_id,customer_id,quantity,price,order_date
8926,54,990,5,146.7,2024-01-20
8927,66,227,1,59.89,2024-01-05
8929,8,753,1,272.6743,2024-01-22
8935,45,368,4,567.23,2024-01-13
8936,37,195,2,209.49,2024-01-12
8937,70,212,4,212.4,2024-01-09
8940,21,340,2,386.3988,2024-01-09
8984,41,350,4,319.13,2024-01-13
9001,21,44,2,302.4,2024-01-20
9010,94,881,5,118.24,2024-01-10


In [0]:
# ============================================================
# SILVER LAYER - CLEAN PRODUCTS
# ============================================================

# Read Products from Bronze
bronze_products_df = spark.table("bronze_products")

print("Bronze product records:", bronze_products_df.count())

# Keep only the current SCD Type 2 records
silver_products_df = (
    bronze_products_df
    .filter(F.col("is_current") == 1)
)

print("Silver product records:", silver_products_df.count())

Bronze product records: 200
Silver product records: 100


In [0]:
# ============================================================
# VERIFY CURRENT PRODUCT RECORDS
# ============================================================

duplicate_current_products = (
    silver_products_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Products with multiple current records:",
    duplicate_current_products.count()
)

Products with multiple current records: 0


In [0]:
# ============================================================
# PRODUCT DATA QUALITY CHECK
# ============================================================

null_product_ids = (
    silver_products_df
    .filter(F.col("product_id").isNull())
    .count()
)

null_prices = (
    silver_products_df
    .filter(F.col("price").isNull())
    .count()
)

invalid_prices = (
    silver_products_df
    .filter(F.col("price") <= 0)
    .count()
)

print("PRODUCT DATA QUALITY")
print("=" * 40)
print("Null product IDs :", null_product_ids)
print("Null prices      :", null_prices)
print("Invalid prices   :", invalid_prices)

PRODUCT DATA QUALITY
Null product IDs : 0
Null prices      : 0
Invalid prices   : 0


In [0]:
# ============================================================
# SAVE SILVER PRODUCTS
# ============================================================

silver_products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_products")

print("silver_products table created successfully.")

silver_products table created successfully.


In [0]:
# ============================================================
# VERIFY SILVER PRODUCTS
# ============================================================

silver_products_check = spark.table("silver_products")

print(
    "Silver Products:",
    silver_products_check.count(),
    "records"
)

display(
    silver_products_check
    .orderBy("product_id")
    .limit(10)
)

Silver Products: 100 records


product_id,price,effective_date,end_date,is_current
1,306.5,2024-01-01,null,1
2,392.46,2024-01-01,null,1
3,555.13,2024-01-01,null,1
4,126.22,2024-01-01,null,1
5,514.02,2024-01-01,null,1
6,175.84,2024-01-01,null,1
7,489.51,2024-01-01,null,1
8,349.27,2024-01-01,null,1
9,414.3,2024-01-01,null,1
10,396.63,2024-01-01,null,1


In [0]:
# ============================================================
# SILVER LAYER - CLEAN PAYMENTS
# ============================================================

# Read Payments from Bronze
bronze_payments_df = spark.table("bronze_payments")

print("Bronze payment records:", bronze_payments_df.count())

# Keep successful payments
silver_payments_df = (
    bronze_payments_df
    .filter(F.col("payment_status") == "SUCCESS")
)

print(
    "Silver payment records:",
    silver_payments_df.count()
)

Bronze payment records: 9300
Silver payment records: 9300


In [0]:
# ============================================================
# PAYMENT DATA QUALITY CHECK
# ============================================================

null_payment_ids = (
    silver_payments_df
    .filter(F.col("payment_id").isNull())
    .count()
)

null_order_ids = (
    silver_payments_df
    .filter(F.col("order_id").isNull())
    .count()
)

invalid_amounts = (
    silver_payments_df
    .filter(F.col("payment_amount") <= 0)
    .count()
)

duplicate_payment_ids = (
    silver_payments_df
    .groupBy("payment_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("PAYMENT DATA QUALITY")
print("=" * 40)
print("Null payment IDs :", null_payment_ids)
print("Null order IDs   :", null_order_ids)
print("Invalid amounts  :", invalid_amounts)
print("Duplicate payment IDs:", duplicate_payment_ids)

PAYMENT DATA QUALITY
Null payment IDs : 0
Null order IDs   : 0
Invalid amounts  : 0
Duplicate payment IDs: 0


In [0]:
# ============================================================
# IDENTIFY ORPHAN PAYMENTS
# ============================================================

orphan_check = (
    silver_payments_df
    .join(
        silver_orders_df.select("order_id"),
        "order_id",
        "left"
    )
    .withColumn(
        "is_orphan",
        F.when(
            F.col("order_id").isNull(),
            1
        ).otherwise(0)
    )
)

In [0]:
# ============================================================
# FIND ORPHAN PAYMENTS
# ============================================================

orphan_payments = (
    silver_payments_df
    .join(
        silver_orders_df.select("order_id"),
        "order_id",
        "left_anti"
    )
)

print(
    "Orphan payments:",
    orphan_payments.count()
)

display(orphan_payments.limit(10))

Orphan payments: 300


order_id,payment_id,payment_amount,payment_status
10001,9001,3546.84,SUCCESS
10002,9002,251.62,SUCCESS
10003,9003,2332.58,SUCCESS
10004,9004,870.92,SUCCESS
10005,9005,4554.18,SUCCESS
10006,9006,591.56,SUCCESS
10007,9007,2826.5,SUCCESS
10008,9008,1213.1,SUCCESS
10009,9009,4453.45,SUCCESS
10010,9010,2003.62,SUCCESS


In [0]:
# ============================================================
# SAVE SILVER PAYMENTS
# ============================================================

silver_payments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_payments")

print("silver_payments table created successfully.")

silver_payments table created successfully.


In [0]:
# ============================================================
# VERIFY SILVER PAYMENTS
# ============================================================

silver_payments_check = spark.table("silver_payments")

print(
    "Silver Payments:",
    silver_payments_check.count(),
    "records"
)

display(
    silver_payments_check.limit(10)
)

Silver Payments: 9300 records


payment_id,order_id,payment_amount,payment_status
1,1,590.4,SUCCESS
2,2,1079.92,SUCCESS
3,3,426.07,SUCCESS
4,4,1106.58,SUCCESS
5,5,1650.65,SUCCESS
6,6,201.2,SUCCESS
7,7,231.52,SUCCESS
8,8,487.92,SUCCESS
9,9,487.92,SUCCESS
10,10,209.49,SUCCESS


In [0]:
# ============================================================
# GOLD LAYER - ORDER LEVEL REVENUE ANALYSIS
# ============================================================

# Load Silver tables
orders = spark.table("silver_orders")
products = spark.table("silver_products")
payments = spark.table("silver_payments")

print("Orders:", orders.count())
print("Products:", products.count())
print("Payments:", payments.count())

Orders: 10000
Products: 100
Payments: 9300


In [0]:
# ============================================================
# JOIN ORDERS WITH CURRENT PRODUCT CATALOG
# ============================================================

order_product_df = (
    orders.alias("o")
    .join(
        products.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "left"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.product_id"),
        F.col("o.customer_id"),
        F.col("o.quantity"),
        F.col("o.price").alias("order_price"),
        F.col("p.price").alias("catalog_price"),
        F.col("o.order_date")
    )
)

print(
    "Order-product records:",
    order_product_df.count()
)

display(order_product_df.limit(10))

Order-product records: 10000


order_id,product_id,customer_id,quantity,order_price,catalog_price,order_date
8926,54,990,5,146.7,146.7,2024-01-20
8927,66,227,1,59.89,59.89,2024-01-05
8929,8,753,1,272.6743,349.27,2024-01-22
8935,45,368,4,567.23,567.23,2024-01-13
8936,37,195,2,209.49,209.49,2024-01-12
8937,70,212,4,212.4,212.4,2024-01-09
8940,21,340,2,386.3988,302.4,2024-01-09
8984,41,350,4,319.13,319.13,2024-01-13
9001,21,44,2,302.4,302.4,2024-01-20
9010,94,881,5,118.24,118.24,2024-01-10


In [0]:
# ============================================================
# CALCULATE REVENUE
# ============================================================

order_product_df = (
    order_product_df
    .withColumn(
        "expected_revenue",
        F.col("catalog_price") * F.col("quantity")
    )
    .withColumn(
        "order_revenue",
        F.col("order_price") * F.col("quantity")
    )
    .withColumn(
        "price_difference",
        F.col("expected_revenue") - F.col("order_revenue")
    )
)

display(
    order_product_df.limit(10)
)

order_id,product_id,customer_id,quantity,order_price,catalog_price,order_date,expected_revenue,order_revenue,price_difference
8926,54,990,5,146.7,146.7,2024-01-20,733.5,733.5,0.0
8927,66,227,1,59.89,59.89,2024-01-05,59.89,59.89,0.0
8929,8,753,1,272.6743,349.27,2024-01-22,349.27,272.6743,76.59569999999997
8935,45,368,4,567.23,567.23,2024-01-13,2268.92,2268.92,0.0
8936,37,195,2,209.49,209.49,2024-01-12,418.98,418.98,0.0
8937,70,212,4,212.4,212.4,2024-01-09,849.6,849.6,0.0
8940,21,340,2,386.3988,302.4,2024-01-09,604.8,772.7976,-167.99760000000003
8984,41,350,4,319.13,319.13,2024-01-13,1276.52,1276.52,0.0
9001,21,44,2,302.4,302.4,2024-01-20,604.8,604.8,0.0
9010,94,881,5,118.24,118.24,2024-01-10,591.1999999999999,591.1999999999999,0.0


In [0]:
# ============================================================
# AGGREGATE PAYMENTS BY ORDER
# ============================================================

payment_summary = (
    payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_amount").alias("total_paid")
    )
)

display(payment_summary.limit(10))

order_id,total_paid
1,590.4
2,1079.92
3,426.07
4,1106.58
5,1650.65
6,201.2
7,231.52
8,487.92
9,487.92
10,209.49


In [0]:
# ============================================================
# JOIN PAYMENT INFORMATION
# ============================================================

gold_orders = (
    order_product_df.alias("o")
    .join(
        payment_summary.alias("p"),
        F.col("o.order_id") == F.col("p.order_id"),
        "left"
    )
    .select(
        F.col("o.*"),
        F.col("p.total_paid")
    )
)

In [0]:
# ============================================================
# DETERMINE PAYMENT STATUS
# ============================================================

gold_orders = (
    gold_orders
    .withColumn(
        "payment_status",
        F.when(
            F.col("total_paid").isNull(),
            "MISSING_PAYMENT"
        )
        .when(
            F.abs(
                F.col("total_paid") -
                F.col("order_revenue")
            ) > 0.01,
            "PAYMENT_MISMATCH"
        )
        .otherwise("PAID")
    )
)

display(gold_orders.limit(20))

order_id,product_id,customer_id,quantity,order_price,catalog_price,order_date,expected_revenue,order_revenue,price_difference,total_paid,payment_status
8926,54,990,5,146.7,146.7,2024-01-20,733.5,733.5,0.0,733.5,PAID
8927,66,227,1,59.89,59.89,2024-01-05,59.89,59.89,0.0,null,MISSING_PAYMENT
8929,8,753,1,272.6743,349.27,2024-01-22,349.27,272.6743,76.59569999999997,272.67,PAID
8935,45,368,4,567.23,567.23,2024-01-13,2268.92,2268.92,0.0,null,MISSING_PAYMENT
8936,37,195,2,209.49,209.49,2024-01-12,418.98,418.98,0.0,418.98,PAID
8937,70,212,4,212.4,212.4,2024-01-09,849.6,849.6,0.0,849.6,PAID
8940,21,340,2,386.3988,302.4,2024-01-09,604.8,772.7976,-167.99760000000003,null,MISSING_PAYMENT
8984,41,350,4,319.13,319.13,2024-01-13,1276.52,1276.52,0.0,1276.52,PAID
9001,21,44,2,302.4,302.4,2024-01-20,604.8,604.8,0.0,604.8,PAID
9010,94,881,5,118.24,118.24,2024-01-10,591.1999999999999,591.1999999999999,0.0,591.2,PAID


In [0]:
# ============================================================
# REVENUE ANOMALY FLAGS
# ============================================================

gold_orders = (
    gold_orders

    # Price mismatch
    .withColumn(
        "price_mismatch_flag",
        F.when(
            F.abs(
                F.col("order_price") -
                F.col("catalog_price")
            ) > 0.01,
            1
        ).otherwise(0)
    )

    # Missing payment
    .withColumn(
        "missing_payment_flag",
        F.when(
            F.col("total_paid").isNull(),
            1
        ).otherwise(0)
    )

    # Shadow revenue
    .withColumn(
        "shadow_revenue",
        F.when(
            F.col("total_paid").isNull(),
            F.col("expected_revenue")
        ).otherwise(0)
    )
)

display(gold_orders.limit(20))

order_id,product_id,customer_id,quantity,order_price,catalog_price,order_date,expected_revenue,order_revenue,price_difference,total_paid,payment_status,price_mismatch_flag,missing_payment_flag,shadow_revenue
8926,54,990,5,146.7,146.7,2024-01-20,733.5,733.5,0.0,733.5,PAID,0,0,0.0
8927,66,227,1,59.89,59.89,2024-01-05,59.89,59.89,0.0,null,MISSING_PAYMENT,0,1,59.89
8929,8,753,1,272.6743,349.27,2024-01-22,349.27,272.6743,76.59569999999997,272.67,PAID,1,0,0.0
8935,45,368,4,567.23,567.23,2024-01-13,2268.92,2268.92,0.0,null,MISSING_PAYMENT,0,1,2268.92
8936,37,195,2,209.49,209.49,2024-01-12,418.98,418.98,0.0,418.98,PAID,0,0,0.0
8937,70,212,4,212.4,212.4,2024-01-09,849.6,849.6,0.0,849.6,PAID,0,0,0.0
8940,21,340,2,386.3988,302.4,2024-01-09,604.8,772.7976,-167.99760000000003,null,MISSING_PAYMENT,1,1,604.8
8984,41,350,4,319.13,319.13,2024-01-13,1276.52,1276.52,0.0,1276.52,PAID,0,0,0.0
9001,21,44,2,302.4,302.4,2024-01-20,604.8,604.8,0.0,604.8,PAID,0,0,0.0
9010,94,881,5,118.24,118.24,2024-01-10,591.1999999999999,591.1999999999999,0.0,591.2,PAID,0,0,0.0


In [0]:
# ============================================================
# REVENUE INTEGRITY SUMMARY
# ============================================================

revenue_summary = (
    gold_orders
    .agg(
        F.count("*").alias("total_orders"),

        F.sum("expected_revenue")
        .alias("expected_revenue"),

        F.sum("order_revenue")
        .alias("order_revenue"),

        F.sum("total_paid")
        .alias("total_collected"),

        F.sum("shadow_revenue")
        .alias("shadow_revenue"),

        F.sum("price_mismatch_flag")
        .alias("price_mismatches"),

        F.sum("missing_payment_flag")
        .alias("missing_payments")
    )
)

display(revenue_summary)

total_orders,expected_revenue,order_revenue,total_collected,shadow_revenue,price_mismatches,missing_payments
10000,8457640.079999913,8444577.259799909,7610516.139999926,834111.8499999981,1014,1000


In [0]:
# ============================================================
# ORPHAN PAYMENT ANALYSIS
# ============================================================

orphan_payments_gold = (
    silver_payments_df
    .join(
        silver_orders_df.select("order_id"),
        "order_id",
        "left_anti"
    )
)

orphan_summary = (
    orphan_payments_gold
    .agg(
        F.count("*").alias("orphan_payment_count"),
        F.sum("payment_amount").alias("orphan_payment_amount")
    )
)

display(orphan_summary)

orphan_payment_count,orphan_payment_amount
300,733785.68


In [0]:
# ============================================================
# FINAL REVENUE INTEGRITY KPIs
# ============================================================

summary_row = revenue_summary.first()
orphan_row = orphan_summary.first()

total_orders = summary_row["total_orders"]
expected_revenue = summary_row["expected_revenue"]
order_revenue = summary_row["order_revenue"]
total_collected = summary_row["total_collected"] or 0
shadow_revenue = summary_row["shadow_revenue"] or 0
price_mismatches = summary_row["price_mismatches"] or 0
missing_payments = summary_row["missing_payments"] or 0

orphan_payment_count = orphan_row["orphan_payment_count"]
orphan_payment_amount = orphan_row["orphan_payment_amount"] or 0

# Revenue leakage percentage
revenue_leakage_percentage = (
    shadow_revenue / expected_revenue
) * 100

# Collection percentage
collection_percentage = (
    total_collected / expected_revenue
) * 100

print("==============================================")
print("       REVENUE INTEGRITY REPORT")
print("==============================================")

print("Total Orders              :", total_orders)
print(
    "Expected Revenue          : ₹",
    builtins.round(expected_revenue, 2)
)
print(
    "Order Revenue             : ₹",
    builtins.round(order_revenue, 2)
)
print(
    "Collected Revenue         : ₹",
    builtins.round(total_collected, 2)
)
print(
    "Shadow Revenue            : ₹",
    builtins.round(shadow_revenue, 2)
)
print("Price Mismatches          :", price_mismatches)
print("Missing Payments          :", missing_payments)
print("Orphan Payments           :", orphan_payment_count)
print(
    "Orphan Payment Amount     : ₹",
    builtins.round(orphan_payment_amount, 2)
)
print(
    "Revenue Leakage %         :",
    builtins.round(revenue_leakage_percentage, 2),
    "%"
)
print(
    "Revenue Collection %      :",
    builtins.round(collection_percentage, 2),
    "%"
)

       REVENUE INTEGRITY REPORT
Total Orders              : 10000
Expected Revenue          : ₹ 8457640.08
Order Revenue             : ₹ 8444577.26
Collected Revenue         : ₹ 7610516.14
Shadow Revenue            : ₹ 834111.85
Price Mismatches          : 1014
Missing Payments          : 1000
Orphan Payments           : 300
Orphan Payment Amount     : ₹ 733785.68
Revenue Leakage %         : 9.86 %
Revenue Collection %      : 89.98 %


In [0]:
# ============================================================
# SAVE GOLD ORDER-LEVEL TABLE
# ============================================================

gold_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_order_revenue_integrity")

print("Gold order-level table created successfully.")

Gold order-level table created successfully.


In [0]:
gold_order_check = spark.table(
    "gold_order_revenue_integrity"
)

print(
    "Gold order records:",
    gold_order_check.count()
)

display(gold_order_check.limit(20))

Gold order records: 10000


order_id,product_id,customer_id,quantity,order_price,catalog_price,order_date,expected_revenue,order_revenue,price_difference,total_paid,payment_status,price_mismatch_flag,missing_payment_flag,shadow_revenue
8926,54,990,5,146.7,146.7,2024-01-20,733.5,733.5,0.0,733.5,PAID,0,0,0.0
8927,66,227,1,59.89,59.89,2024-01-05,59.89,59.89,0.0,null,MISSING_PAYMENT,0,1,59.89
8929,8,753,1,272.6743,349.27,2024-01-22,349.27,272.6743,76.59569999999997,272.67,PAID,1,0,0.0
8935,45,368,4,567.23,567.23,2024-01-13,2268.92,2268.92,0.0,null,MISSING_PAYMENT,0,1,2268.92
8936,37,195,2,209.49,209.49,2024-01-12,418.98,418.98,0.0,418.98,PAID,0,0,0.0
8937,70,212,4,212.4,212.4,2024-01-09,849.6,849.6,0.0,849.6,PAID,0,0,0.0
8940,21,340,2,386.3988,302.4,2024-01-09,604.8,772.7976,-167.99760000000003,null,MISSING_PAYMENT,1,1,604.8
8984,41,350,4,319.13,319.13,2024-01-13,1276.52,1276.52,0.0,1276.52,PAID,0,0,0.0
9001,21,44,2,302.4,302.4,2024-01-20,604.8,604.8,0.0,604.8,PAID,0,0,0.0
9010,94,881,5,118.24,118.24,2024-01-10,591.1999999999999,591.1999999999999,0.0,591.2,PAID,0,0,0.0


In [0]:
# ============================================================
# ANOMALY SUMMARY
# ============================================================

anomaly_summary = spark.createDataFrame(
    [
        (
            "PRICE_MISMATCH",
            price_mismatches,
            None
        ),
        (
            "MISSING_PAYMENT",
            missing_payments,
            shadow_revenue
        ),
        (
            "ORPHAN_PAYMENT",
            orphan_payment_count,
            orphan_payment_amount
        )
    ],
    [
        "anomaly_type",
        "record_count",
        "financial_impact"
    ]
)

display(anomaly_summary)

anomaly_type,record_count,financial_impact
PRICE_MISMATCH,1014,null
MISSING_PAYMENT,1000,834111.8499999981
ORPHAN_PAYMENT,300,733785.68


In [0]:
# ============================================================
# SAVE GOLD ANOMALY TABLE
# ============================================================

anomaly_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_revenue_anomalies")

print("Gold anomaly table created successfully.")

Gold anomaly table created successfully.


In [0]:
# ============================================================
# REVENUE INTEGRITY SCORE
# ============================================================

summary = revenue_summary.first()

expected_revenue = summary["expected_revenue"]
shadow_revenue = summary["shadow_revenue"]
price_mismatches = summary["price_mismatches"]
missing_payments = summary["missing_payments"]

# Revenue leakage percentage
leakage_percentage = (
    shadow_revenue / expected_revenue
) * 100

# Integrity score
integrity_score = 100 - leakage_percentage

if integrity_score < 0:
    integrity_score = 0

print("==============================================")
print("        REVENUE INTEGRITY SCORECARD")
print("==============================================")

print(
    "Expected Revenue       : ₹",
    builtins.round(expected_revenue, 2)
)

print(
    "Shadow Revenue         : ₹",
    builtins.round(shadow_revenue, 2)
)

print(
    "Revenue Leakage        :",
    builtins.round(leakage_percentage, 2),
    "%"
)

print(
    "Price Mismatches       :",
    price_mismatches
)

print(
    "Missing Payments       :",
    missing_payments
)

print(
    "Integrity Score        :",
    builtins.round(integrity_score, 2),
    "/ 100"
)

        REVENUE INTEGRITY SCORECARD
Expected Revenue       : ₹ 8457640.08
Shadow Revenue         : ₹ 834111.85
Revenue Leakage        : 9.86 %
Price Mismatches       : 1014
Missing Payments       : 1000
Integrity Score        : 90.14 / 100


In [0]:
# ============================================================
# FINAL REVENUE INTEGRITY REPORT
# ============================================================

summary = revenue_summary.first()

report_data = [
    (
        "Revenue",
        "Expected Revenue",
        float(summary["expected_revenue"])
    ),
    (
        "Revenue",
        "Order Revenue",
        float(summary["order_revenue"])
    ),
    (
        "Revenue",
        "Collected Revenue",
        float(summary["total_collected"])
    ),
    (
        "Revenue Leakage",
        "Shadow Revenue",
        float(summary["shadow_revenue"])
    ),
    (
        "Revenue Leakage",
        "Price Mismatches",
        float(summary["price_mismatches"])
    ),
    (
        "Revenue Leakage",
        "Missing Payments",
        float(summary["missing_payments"])
    ),
    (
        "Payment Integrity",
        "Orphan Payments",
        float(orphan_payment_count)
    ),
    (
        "Payment Integrity",
        "Orphan Payment Amount",
        float(orphan_payment_amount)
    ),
    (
        "Score",
        "Revenue Integrity Score",
        float(integrity_score)
    )
]

revenue_integrity_report = spark.createDataFrame(
    report_data,
    ["category", "metric", "value"]
)

display(revenue_integrity_report)

category,metric,value
Revenue,Expected Revenue,8457640.079999913
Revenue,Order Revenue,8444577.259799909
Revenue,Collected Revenue,7610516.139999926
Revenue Leakage,Shadow Revenue,834111.8499999981
Revenue Leakage,Price Mismatches,1014.0
Revenue Leakage,Missing Payments,1000.0
Payment Integrity,Orphan Payments,300.0
Payment Integrity,Orphan Payment Amount,733785.68
Score,Revenue Integrity Score,90.13777079527831


In [0]:
# ============================================================
# SAVE FINAL REVENUE INTEGRITY REPORT
# ============================================================

revenue_integrity_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_revenue_integrity_report")

print("Final Revenue Integrity Report saved successfully.")

Final Revenue Integrity Report saved successfully.
